# Generation, oversampling, and traceability

Generate synthetic rows and inspect their trace records.

In [ ]:
import numpy as np
import pandas as pd
from mimic import MIMIC, GenerationPolicy, RandomForestPathEncoder, MixedFeatureDecoder

rng = np.random.default_rng(2)
major = pd.DataFrame({
    "x": rng.normal(0, 1, 120),
    "y": rng.normal(0, 1, 120),
    "label": "majority",
})
minor = pd.DataFrame({
    "x": rng.normal(2.5, 0.5, 20),
    "y": rng.normal(2.5, 0.5, 20),
    "label": "minority",
})
df = pd.concat([major, minor], ignore_index=True)
df["id"] = np.arange(len(df))
df["label"].value_counts()

In [ ]:
mimic = MIMIC(
    ignore_columns=["id"],
    regression_columns=["x", "y"],
    classification_columns=["label"],
    encoder=RandomForestPathEncoder(n_estimators=20, embedding_dim=6, random_state=2),
    decoder=MixedFeatureDecoder.random_forest(n_estimators=20, random_state=2),
    policy=GenerationPolicy(method="smote", neighbour_mode="normal", n_neighbors=5, lambda_range=(0.0, 1.0)),
    n_bootstrap=2,
    random_state=2,
)
mimic.fit(df)
synthetic, trace = mimic.sample(12, return_trace=True)
synthetic.head(), trace.head()

In [ ]:
minority_needed = df["label"].value_counts()["majority"] - df["label"].value_counts()["minority"]
generated, generation_trace = mimic.sample(minority_needed, return_trace=True)
minority_synthetic = generated[generated["label"] == "minority"]
balanced = pd.concat([df.drop(columns=["id"]), minority_synthetic], ignore_index=True)
balanced["label"].value_counts(), generation_trace.head()

In [ ]:
disp = MIMIC(
    ignore_columns=["id"],
    regression_columns=["x", "y"],
    classification_columns=["label"],
    encoder=RandomForestPathEncoder(n_estimators=20, embedding_dim=6, random_state=3),
    decoder=MixedFeatureDecoder.random_forest(n_estimators=20, random_state=3),
    policy=GenerationPolicy(method="displacement", neighbour_mode="normal", n_neighbors=5, lambda_range=(0.0, 1.0)),
    n_bootstrap=2,
    random_state=3,
)
disp.fit(df)
disp_synthetic, disp_trace = disp.sample(5, return_trace=True)
disp_synthetic, disp_trace